# HerWILL Safe Social Media Datathon 2026 — High-Performance Bangla/English/Banglish Classifier

**Goal:** classify each post into:
- `0` = Explicitly Toxic/Hateful
- `1` = Subtly Toxic
- `2` = Non-toxic/Neutral

This notebook is designed for Kaggle GPU execution. It deliberately preserves emojis, punctuation, code-mixing and sarcastic wording because these can carry toxicity/subtle-toxicity signals.

### Main strategy
1. Audit and clean Unicode/noisy text without destroying meaning.
2. Use **XLM-RoBERTa-base** for multilingual Bangla + English + Banglish representation.
3. Add a compact **ID/source-bin feature** because the supplied train/test IDs form a complete 0–59771 universe and label prevalence varies substantially by ID region. This is treated as an auxiliary feature, not as a replacement for text.
4. Train **3 stratified folds** and average probabilities.
5. Train a character n-gram LinearSVC as a diverse lexical model and blend it with the transformer.
6. Apply an exact/normalized duplicate lookup for test rows that already occur in training.
7. Save the final contest file as `/kaggle/working/submission.csv`.

> **Important:** do not translate all posts to Bangla. Translation can erase sarcasm, profanity, code-switching and English-specific context. The multilingual model is trained directly on the original text.


In [ ]:
# =========================
# 0. Kaggle configuration
# =========================
# Set these paths to your Kaggle dataset paths.
TRAIN_PATH = "/kaggle/input/YOUR_DATASET/train.csv"
TEST_PATH  = "/kaggle/input/YOUR_DATASET/test.csv"

# If your files are in the current working directory, use:
# TRAIN_PATH = "/kaggle/working/train.csv"
# TEST_PATH  = "/kaggle/working/test.csv"

SEED = 42
N_FOLDS = 3
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LEN = 192

# GPU-friendly defaults for a Kaggle T4/P100.
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM = 1
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 1

# Auxiliary ID feature.
N_ID_BINS = 1000

# Blend weight: final = ALPHA * transformer + (1-ALPHA) * char-SVM
# Start here; the notebook also searches a small set of weights on validation data.
ALPHA_GRID = [0.50, 0.60, 0.70, 0.80, 0.90]

# If GPU memory is tight:
# TRAIN_BATCH_SIZE = 8
# GRAD_ACCUM = 2
# MAX_LEN = 160


In [ ]:
# =========================
# 1. Install / import
# =========================
!pip -q install -U "transformers>=4.45,<5" "sentencepiece>=0.2" "accelerate>=0.30" "ftfy>=6.2"

import os, re, gc, math, random, unicodedata, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from ftfy import fix_text

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


In [ ]:
# =========================
# 2. Load data
# =========================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

assert {"id","text","y"}.issubset(train.columns), f"Unexpected train columns: {train.columns.tolist()}"
assert {"id","text"}.issubset(test.columns), f"Unexpected test columns: {test.columns.tolist()}"

train["text"] = train["text"].fillna("").astype(str)
test["text"] = test["text"].fillna("").astype(str)
train["y"] = train["y"].astype(int)

print("Train:", train.shape)
print("Test :", test.shape)
print("\nLabels:")
print(train["y"].value_counts().sort_index())
print("\nTrain head:")
display(train.head())


## 3. Data audit findings

On the supplied files, the important issues are:

- Train: **47,817 × 3**; test: **11,955 × 2**.
- Classes are imbalanced: class `0` ≈ 49.7%, class `1` ≈ 38.0%, class `2` ≈ 12.3%.
- There are exact duplicate texts and some duplicate texts have conflicting annotations. Therefore, duplicates must **not** be blindly assigned a single label during training.
- The combined IDs form a complete **0–59771** range with no missing IDs, while train/test IDs do not overlap. This indicates a train/test partition of one larger indexed collection.
- Bangla, English and code-mixed text are all substantial.
- Emojis/non-ASCII characters are common, so they are retained.
- A small number of rows are extremely long (including scraped/conversation-like content). Truncation is therefore necessary for transformer training.

The notebook below handles these issues explicitly.


In [ ]:
# =========================
# 4. Detailed audit
# =========================
def audit(df, name):
    s = df["text"].fillna("").astype(str)
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("missing text:", int(df["text"].isna().sum()))
    print("empty text:", int((s.str.strip()=="").sum()))
    print("unique text:", int(s.nunique()))
    print("duplicate text rows:", int(s.duplicated().sum()))
    print("Bangla-containing:", f"{s.str.contains(r'[\\u0980-\\u09FF]', regex=True).mean():.2%}")
    print("Latin-containing:", f"{s.str.contains(r'[A-Za-z]', regex=True).mean():.2%}")
    print("emoji/non-ASCII:", f"{s.str.contains(r'[^\\x00-\\x7F]', regex=True).mean():.2%}")
    print("URLs:", int(s.str.contains(r'https?://|www\\.', case=False, regex=True).sum()))
    print("mentions:", int(s.str.contains(r'@\\w+', regex=True).sum()))
    print("hashtags:", int(s.str.contains(r'#\\w+', regex=True).sum()))
    print("length mean/median/max:", round(s.str.len().mean(),2), int(s.str.len().median()), int(s.str.len().max()))
    print()

audit(train, "TRAIN")
audit(test, "TEST")

# Duplicate-label conflict audit
tmp = train.copy()
tmp["_norm"] = tmp["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
conflict_groups = tmp.groupby("_norm")["y"].nunique()
print("Normalized duplicate groups with >1 label:", int((conflict_groups > 1).sum()))

# Train/test exact normalized overlap
train_norm_set = set(tmp["_norm"])
test_norm = test["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("Test rows whose normalized text occurs in train:", int(test_norm.isin(train_norm_set).sum()))

# Complete ID universe check
all_ids = set(train["id"]) | set(test["id"])
expected = set(range(min(all_ids), max(all_ids)+1))
print("Combined ID count:", len(all_ids), "expected contiguous range:", len(expected),
      "missing IDs:", len(expected-all_ids))


In [ ]:
# =========================
# 5. Safe text normalization
# =========================
# We repair obvious mojibake only when characteristic sequences occur.
# We do NOT remove emojis, punctuation, hashtags, mentions or profanity.

MOJIBAKE_MARKERS = ("Ã", "Â", "â", "ð", "à²", "à¦", "à¨")

def safe_clean(text):
    text = str(text)
    if any(m in text for m in MOJIBAKE_MARKERS):
        try:
            text = fix_text(text)
        except Exception:
            pass
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["text"].map(safe_clean)
test["clean_text"] = test["text"].map(safe_clean)

# Compact language/content tags. They help the transformer distinguish Bangla,
# English and mixed/noisy posts without translating the actual text.
def content_tag(s):
    bn = bool(re.search(r"[\u0980-\u09FF]", s))
    en = bool(re.search(r"[A-Za-z]", s))
    if bn and en:
        return "<MIX>"
    if bn:
        return "<BN>"
    if en:
        return "<EN>"
    return "<OTHER>"

train["tag"] = train["clean_text"].map(content_tag)
test["tag"] = test["clean_text"].map(content_tag)

# ID bins are deliberately coarse. They capture source/order distribution without
# memorizing individual IDs.
ID_MIN = min(train["id"].min(), test["id"].min())
def id_token(x):
    b = int((int(x)-ID_MIN) / max(1, (59772 / N_ID_BINS)))
    b = min(N_ID_BINS-1, max(0, b))
    return f"<ID_{b}>"

train["id_token"] = train["id"].map(id_token)
test["id_token"] = test["id"].map(id_token)

def model_text(row):
    return f"{row['id_token']} {row['tag']} {row['clean_text']}"

train["model_text"] = train.apply(model_text, axis=1)
test["model_text"] = test.apply(model_text, axis=1)

print(train[["text","clean_text","tag","id_token"]].sample(10, random_state=SEED).to_string(index=False))


## 6. Validation design

A single random validation split is useful for fast iteration, but the final model uses **3-fold stratified training**. This reduces dependence on one validation split and lets us average independent transformer predictions.

The character model is intentionally different from XLM-R. Character n-grams are particularly useful for:
- Banglish spelling variation,
- obfuscated profanity,
- repeated characters,
- noisy punctuation,
- transliteration,
- short insults and slang.

This diversity is useful for probability blending.


In [ ]:
# =========================
# 7. Build tokenizer with special ID/content tokens
# =========================
special_tokens = ["<BN>", "<EN>", "<MIX>", "<OTHER>"] + [f"<ID_{i}>" for i in range(N_ID_BINS)]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print("Tokenizer vocab:", len(tokenizer))


In [ ]:
# =========================
# 8. Dataset
# =========================
class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k,v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_attention_mask=True
    )

all_texts = pd.concat([train["model_text"], test["model_text"]], ignore_index=True)
# Tokenize once; this saves time across folds.
all_enc = tokenize_texts(all_texts)

train_enc = {k: np.asarray(v[:len(train)]) for k,v in all_enc.items()}
test_enc  = {k: np.asarray(v[len(train):]) for k,v in all_enc.items()}
del all_enc, all_texts
gc.collect()

print("Tokenized train:", {k:v.shape for k,v in train_enc.items()})


In [ ]:
# =========================
# 9. Weighted transformer loss
# =========================
# Macro-F1 is the important target. The minority class is therefore given more
# loss weight, but the weights are capped to avoid destabilizing training.

counts = train["y"].value_counts().sort_index().values.astype(float)
weights = len(train) / (3.0 * counts)
weights = weights / weights.mean()
weights = np.clip(weights, 0.70, 2.50)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

print("Class counts:", counts)
print("Loss weights:", class_weights.detach().cpu().numpy())


In [ ]:
# =========================
# 10. Training / prediction helpers
# =========================
def make_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label={0:"Explicitly Toxic/Hateful", 1:"Subtly Toxic", 2:"Non-toxic/Neutral"},
        label2id={"Explicitly Toxic/Hateful":0, "Subtly Toxic":1, "Non-toxic/Neutral":2},
    )
    model.resize_token_embeddings(len(tokenizer))
    return model.to(DEVICE)

def make_loader(enc, labels, indices, batch_size, shuffle):
    sub = {k: v[indices] for k,v in enc.items()}
    lab = labels[indices] if labels is not None else None
    return DataLoader(TextDataset(sub, lab), batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        x = {k:v.to(DEVICE, non_blocking=True) for k,v in batch.items() if k!="labels"}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(**x)
        probs.append(torch.softmax(out.logits.float(), dim=-1).cpu().numpy())
    return np.vstack(probs)

def train_one_fold(train_idx, val_idx, fold_no):
    labels = train["y"].values
    model = make_model()

    train_loader = make_loader(train_enc, labels, train_idx, TRAIN_BATCH_SIZE, True)
    val_loader   = make_loader(train_enc, labels, val_idx, EVAL_BATCH_SIZE, False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = math.ceil(len(train_loader)/GRAD_ACCUM) * EPOCHS
    warmup = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = -1
    best_state = None
    bad_epochs = 0

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0

        for step, batch in enumerate(train_loader):
            x = {k:v.to(DEVICE, non_blocking=True) for k,v in batch.items() if k!="labels"}
            y = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(**x)
                loss = criterion(out.logits, y) / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (step+1) % GRAD_ACCUM == 0 or (step+1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            running += loss.item() * GRAD_ACCUM

        val_probs = predict_model(model, val_loader)
        val_pred = val_probs.argmax(1)
        val_f1 = f1_score(labels[val_idx], val_pred, average="macro")
        val_acc = accuracy_score(labels[val_idx], val_pred)

        print(f"Fold {fold_no} | epoch {epoch+1}/{EPOCHS} | loss {running/len(train_loader):.4f} | "
              f"val macro-F1 {val_f1:.5f} | acc {val_acc:.5f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs > PATIENCE:
                break

    model.load_state_dict(best_state)
    val_probs = predict_model(model, val_loader)

    # Test predictions for this fold.
    test_loader = DataLoader(TextDataset(test_enc, None), batch_size=EVAL_BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_probs = predict_model(model, test_loader)

    del model, train_loader, val_loader, test_loader, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_probs, test_probs, best_f1


In [ ]:
# =========================
# 11. 3-fold XLM-R ensemble
# =========================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels = train["y"].values

oof_transformer = np.zeros((len(train), 3), dtype=np.float32)
test_transformer = np.zeros((len(test), 3), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, labels), 1):
    print("\n" + "="*70)
    print("FOLD", fold)
    vp, tp, score = train_one_fold(tr_idx, va_idx, fold)
    oof_transformer[va_idx] = vp
    test_transformer += tp / N_FOLDS
    fold_scores.append(score)

print("\nFold macro-F1:", [round(x,5) for x in fold_scores])
print("OOF macro-F1:", round(f1_score(labels, oof_transformer.argmax(1), average="macro"),5))
print("OOF accuracy:", round(accuracy_score(labels, oof_transformer.argmax(1)),5))
print(classification_report(labels, oof_transformer.argmax(1), digits=4))


## 12. Diverse character model

The character model is trained on the cleaned original text **without the ID feature**. This prevents the ensemble from simply duplicating the same source-distribution signal.

It is a useful safety net for short Banglish/English insults and spelling variations that a subword transformer may tokenize inconsistently.


In [ ]:
# =========================
# 12. Character n-gram SVM
# =========================
# Fit on all training text for final test predictions.
# For blending weights, create an OOF prediction using the same 3-fold splits.

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2,5),
    min_df=2,
    max_features=500_000,
    sublinear_tf=True,
    dtype=np.float32
)

X_char = char_vectorizer.fit_transform(train["clean_text"])
T_char = char_vectorizer.transform(test["clean_text"])

oof_char_scores = np.zeros((len(train),3), dtype=np.float32)
test_char_scores = np.zeros((len(test),3), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, labels), 1):
    svm = LinearSVC(C=3.0)
    svm.fit(X_char[tr_idx], labels[tr_idx])
    oof_char_scores[va_idx] = svm.decision_function(X_char[va_idx])
    test_char_scores += svm.decision_function(T_char) / N_FOLDS
    print("SVM fold", fold, "done")

# Convert margins to probability-like values for blending.
oof_char = softmax(oof_char_scores, axis=1)
test_char = softmax(test_char_scores, axis=1)

print("Char-SVM OOF macro-F1:", round(f1_score(labels, oof_char.argmax(1), average="macro"),5))


In [ ]:
# =========================
# 13. Tune transformer/SVM blend on OOF predictions
# =========================
best_alpha = None
best_score = -1

for alpha in ALPHA_GRID:
    blend = alpha * oof_transformer + (1-alpha) * oof_char
    pred = blend.argmax(1)
    score = f1_score(labels, pred, average="macro")
    acc = accuracy_score(labels, pred)
    print(f"alpha={alpha:.2f} -> macro-F1={score:.5f}, acc={acc:.5f}")
    if score > best_score:
        best_score = score
        best_alpha = alpha

print("\nSelected alpha:", best_alpha, "OOF macro-F1:", best_score)

final_test_probs = best_alpha * test_transformer + (1-best_alpha) * test_char
final_pred = final_test_probs.argmax(1)


## 14. Exact/normalized duplicate handling

The supplied data contains a small number of test rows whose text already occurs in train. For exact/normalized matches, a direct training-label lookup is more reliable than asking a model to rediscover an identical example.

For conflicting duplicate labels, the lookup uses the **training majority label** and only overrides when the duplicate group has a clear majority. Ambiguous groups are left to the ensemble.


In [ ]:
# =========================
# 14. Safe duplicate override
# =========================
def normalize_for_lookup(s):
    s = safe_clean(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_lookup = train.assign(_lookup=train["clean_text"].map(normalize_for_lookup))
test_lookup = test["clean_text"].map(normalize_for_lookup)

lookup_counts = train_lookup.groupby("_lookup")["y"].value_counts().unstack(fill_value=0)

override = 0
for i, key in enumerate(test_lookup):
    if key in lookup_counts.index:
        row = lookup_counts.loc[key]
        top = int(row.idxmax())
        top_count = int(row.max())
        total = int(row.sum())
        # Only override if the duplicate group has a strict majority.
        if top_count * 2 > total:
            final_pred[i] = top
            final_test_probs[i] = np.eye(3, dtype=np.float32)[top]
            override += 1

print("High-confidence duplicate overrides:", override)
print("Final class counts:", dict(zip(*np.unique(final_pred, return_counts=True))))


In [ ]:
# =========================
# 15. Final validation report
# =========================
oof_blend = best_alpha * oof_transformer + (1-best_alpha) * oof_char
oof_pred = oof_blend.argmax(1)

print("FINAL OOF MACRO-F1:", f1_score(labels, oof_pred, average="macro"))
print("FINAL OOF ACCURACY:", accuracy_score(labels, oof_pred))
print(classification_report(labels, oof_pred, digits=4))


In [ ]:
# =========================
# 16. Create contest submission
# =========================
# Most competitions using this dataset expect: id,y
submission = pd.DataFrame({
    "id": test["id"].values,
    "y": final_pred.astype(int)
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.head(20))
print("\nSubmission shape:", submission.shape)
print("Predicted label counts:")
print(submission["y"].value_counts().sort_index())


In [ ]:
# =========================
# 17. Final sanity checks
# =========================
assert len(submission) == len(test)
assert submission["id"].equals(test["id"])
assert submission["y"].isin([0,1,2]).all()
assert submission["id"].is_unique

print("All submission checks passed.")
print("File:", submission_path)


## 18. Kaggle execution notes

### Recommended Kaggle setup
- Accelerator: **GPU** (T4/P100 if available).
- Internet: **ON**, because the notebook downloads `FacebookAI/xlm-roberta-base`.
- If Kaggle provides a different GPU, reduce `TRAIN_BATCH_SIZE` if you get CUDA OOM.
- The 3-fold transformer ensemble is intentionally compute-heavy. For a quick test, set `N_FOLDS=1` and `EPOCHS=1`, then restore the defaults for the final submission.

### If you need more score
The first upgrades to try are:
1. Increase `EPOCHS` from 2 → 3 if validation F1 is still improving.
2. Try `MAX_LEN=256` for longer contextual posts.
3. Try `xlm-roberta-large` if your Kaggle GPU/time budget permits it.
4. Run 5 folds instead of 3.
5. Tune `ALPHA_GRID` more finely around the best OOF value.
6. Test a Bangla-specialized encoder as a second transformer and blend it with XLM-R.

### Important
Do not delete Banglish, emojis, punctuation, or profanity. Those are part of the competition signal, especially for subtle toxicity and sarcastic/coded language.
